In [ ]:
# Per-class analysis of MUSK evaluation results
# Compare trimodal vs bimodal model performance at the class level
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re
print('Starting per-class MUSK analysis...')

In [ ]:
snakemake.input

In [ ]:
# Load MUSK evaluation results for comparison
# Expected inputs: results from trimodal and bimodal_matching models on pannuke and skin datasets


"""Load MUSK evaluation results from JSON files"""
results = []

for result_file, model_type in zip(snakemake.input, snakemake.params.model_types):
    try:
        with open(result_file, 'r') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    data['result_file'] = str(result_file)
                    data['model_type'] = model_type
                    results.append(data)
    except Exception as e:
        print(f'Error loading {result_file}: {e}')


print(f'Loaded {len(results)} evaluation results')

In [ ]:
# Extract per-class metrics and organize by model type and dataset

per_class_data = []

for result in results:
    dataset = result.get('dataset')
    model_path = result.get('pretrained')
    metrics = result.get('metrics')
    class_names = result.get('class_names')
    model_type=result.get('model_type')
    
    # Extract per-class metrics
    for metric_name, metric_value in metrics.items():
        if metric_name.startswith('class_'):
            # Parse class metric name: class_{i}_{class_name}_{metric_type}
            parts = metric_name.split('_', 3)
            if len(parts) >= 4:
                class_idx = int(parts[1])
                class_name = parts[2]
                metric_type = parts[3]
                
                per_class_data.append({
                    'dataset': dataset,
                    'model_type': model_type,
                    'class_idx': class_idx,
                    'class_name': class_name,
                    'metric_type': metric_type,
                    'metric_value': metric_value
                })

per_class_df = pd.DataFrame(per_class_data)
print(f'Extracted {len(per_class_df)} per-class metric entries')
print(per_class_df.head())

In [ ]:
# Create comparison between trimodal and bimodal models

"""Create side-by-side comparison of model performance"""
# Pivot to get model types as columns
comparison_df = per_class_df.pivot_table(
    index=['dataset', 'class_name', 'metric_type'],
    columns='model_type',
    values='metric_value',
    aggfunc='mean'  # In case of duplicates
).reset_index()

# Calculate improvement (trimodal - bimodal)
if 'trimodal' in comparison_df.columns and 'bimodal_matching' in comparison_df.columns:
    comparison_df['improvement'] = comparison_df['trimodal'] - comparison_df['bimodal_matching']
    comparison_df['relative_improvement'] = (
        comparison_df['improvement'] / comparison_df['bimodal_matching'] * 100
    )

model_comparison = comparison_df
print('Model comparison created:')
print(model_comparison.head(10))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Per-Class Performance: Trimodal vs Bimodal Models', fontsize=16)

# Focus on F1 and precision metrics for main comparison
f1_data = model_comparison[model_comparison['metric_type'] == 'f1']
precision_data = model_comparison[model_comparison['metric_type'] == 'precision']

# Plot 1: F1 score comparison for pannuke
pannuke_f1 = f1_data[f1_data['dataset'] == 'pannuke']
if not pannuke_f1.empty:
    x = np.arange(len(pannuke_f1))
    width = 0.35
    axes[0,0].bar(x - width/2, pannuke_f1['bimodal_matching'], width, 
                 label='Bimodal', alpha=0.8)
    axes[0,0].bar(x + width/2, pannuke_f1['trimodal'], width, 
                 label='Trimodal', alpha=0.8)
    axes[0,0].set_xlabel('Classes')
    axes[0,0].set_ylabel('F1 Score')
    axes[0,0].set_title('Pannuke Dataset - F1 Score by Class')
    axes[0,0].set_xticks(x)
    axes[0,0].set_xticklabels(pannuke_f1['class_name'], rotation=45, ha='right')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

# Plot 2: F1 score comparison for skin
skin_f1 = f1_data[f1_data['dataset'] == 'skin']
if not skin_f1.empty:
    x = np.arange(len(skin_f1))
    width = 0.35
    axes[0,1].bar(x - width/2, skin_f1['bimodal_matching'], width, 
                 label='Bimodal', alpha=0.8)
    axes[0,1].bar(x + width/2, skin_f1['trimodal'], width, 
                 label='Trimodal', alpha=0.8)
    axes[0,1].set_xlabel('Classes')
    axes[0,1].set_ylabel('F1 Score')
    axes[0,1].set_title('Skin Dataset - F1 Score by Class')
    axes[0,1].set_xticks(x)
    axes[0,1].set_xticklabels(skin_f1['class_name'], rotation=45, ha='right')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)

# Plot 3: Improvement heatmap
improvement_pivot = model_comparison.pivot_table(
    index='class_name',
    columns=['dataset', 'metric_type'],
    values='relative_improvement',
    aggfunc='mean'
)

sns.heatmap(improvement_pivot.replace([np.inf, -np.inf], np.nan).fillna(0), annot=True, cmap='RdYlBu_r', center=0,
           ax=axes[1,0], fmt='.1f', cbar_kws={'label': 'Relative Improvement (%)'})
axes[1,0].set_title('Relative Improvement: Trimodal vs Bimodal (%)')
axes[1,0].set_xlabel('Dataset & Metric')
axes[1,0].set_ylabel('Class')

# Plot 4: Classes with highest improvement
f1_improvement = f1_data[['dataset', 'class_name', 'improvement']].copy()
f1_improvement = f1_improvement.sort_values('improvement', ascending=False)

if not f1_improvement.empty:
    top_classes = f1_improvement.head(10)
    y_pos = np.arange(len(top_classes))
    colors = ['red' if x < 0 else 'green' for x in top_classes['improvement']]
    
    axes[1,1].barh(y_pos, top_classes['improvement'], color=colors, alpha=0.7)
    axes[1,1].set_yticks(y_pos)
    axes[1,1].set_yticklabels([f"{row['dataset']}: {row['class_name']}" 
                              for _, row in top_classes.iterrows()])
    axes[1,1].set_xlabel('F1 Score Improvement')
    axes[1,1].set_title('Top Classes: F1 Score Improvement\n(Trimodal - Bimodal)')
    axes[1,1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
    axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig.savefig(snakemake.output.plot)

In [ ]:
# Statistical analysis of improvements
print('=== STATISTICAL SUMMARY ===')

# Overall improvement statistics
f1_improvements = model_comparison[
    model_comparison['metric_type'] == 'f1'
]['improvement'].dropna()

if len(f1_improvements) > 0:
    print(f'\nF1 Score Improvements:')
    print(f'Mean improvement: {f1_improvements.mean():.4f}')
    print(f'Median improvement: {f1_improvements.median():.4f}')
    print(f'Std deviation: {f1_improvements.std():.4f}')
    print(f'Classes with positive improvement: {(f1_improvements > 0).sum()}/{len(f1_improvements)}')
    print(f'Max improvement: {f1_improvements.max():.4f}')
    print(f'Min improvement: {f1_improvements.min():.4f}')

# Dataset-specific analysis
for dataset in model_comparison['dataset'].unique():
    dataset_data = model_comparison[
        (model_comparison['dataset'] == dataset) & 
        (model_comparison['metric_type'] == 'f1')
    ]
    
    if not dataset_data.empty:
        improvements = dataset_data['improvement'].dropna()
        positive_improvements = (improvements > 0).sum()
        
        print(f'\n{dataset.upper()} Dataset:')
        print(f'  Mean F1 improvement: {improvements.mean():.4f}')
        print(f'  Classes improved: {positive_improvements}/{len(improvements)}')
        
        # Top improving classes
        top_improved = dataset_data.nlargest(3, 'improvement')
        print(f'  Top improving classes:')
        for _, row in top_improved.iterrows():
            print(f'    {row["class_name"]}: +{row["improvement"]:.4f}')

# Save detailed results
if 'snakemake' in globals() and hasattr(snakemake, 'output'):
    output_path = Path(snakemake.output[0]) if snakemake.output else Path('per_class_analysis.csv')
else:
    output_path = Path('per_class_analysis.csv')

model_comparison.to_csv(output_path, index=False)
print(f'\nDetailed results saved to: {output_path}')